In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/financial_fraud_detection_10k.csv')

# Display the first 5 rows
display(df.head())

,transaction_id,timestamp,customer_id,account_age_days,transaction_amount_usd,payment_method,channel,merchant_category,location_country,device_ip,unusual_delay_flag,received_fake_sms,received_phishing_email,received_fraud_call,is_fraud
0,TXN_100001,2026-09-06 07:53:23,CUST_24592,52,26.12,WIRE_TRANSFER,WEB_PORTAL,RETAIL,CANADA,189.52.44.152,0,0,1,0,0
1,TXN_100002,2026-01-11 07:19:59,CUST_83563,408,140.46,CRYPTO_TRANSFER,ATM,RETAIL,BRAZIL,151.142.3.195,1,0,0,0,0
2,TXN_100003,2026-05-11 17:17:18,CUST_23396,190,64.25,P2P_TRANSFER,MOBILE_APP,CRYPTO_EXCHANGE,RUSSIA,155.135.22.187,1,0,0,0,0
3,TXN_100004,2026-04-24 20:37:10,CUST_92397,1267,46.08,WIRE_TRANSFER,WEB_PORTAL,ELECTRONICS,USA,170.116.148.253,0,1,0,0,0
4,TXN_100005,2026-09-04 21:00:30,CUST_57819,334,12.63,WIRE_TRANSFER,POS_TERMINAL,RETAIL,NIGERIA,180.36.87.137,0,0,0,0,0


In [3]:
# Print the shape of the dataset
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Display data types and non-null counts
df.info()

Dataset Shape: 10000 rows, 15 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           10000 non-null  object 
 1   timestamp                10000 non-null  object 
 2   customer_id              10000 non-null  object 
 3   account_age_days         10000 non-null  int64  
 4   transaction_amount_usd   10000 non-null  float64
 5   payment_method           10000 non-null  object 
 6   channel                  10000 non-null  object 
 7   merchant_category        10000 non-null  object 
 8   location_country         10000 non-null  object 
 9   device_ip                10000 non-null  object 
 10  unusual_delay_flag       10000 non-null  int64  
 11  received_fake_sms        10000 non-null  int64  
 12  received_phishing_email  10000 non-null  int64  
 13  received_fraud_call      10000 non-null

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# 1. Feature Engineering on the main dataset
df_prep = df.copy()
df_prep['timestamp'] = pd.to_datetime(df_prep['timestamp'])
df_prep['hour_of_day'] = df_prep['timestamp'].dt.hour
df_prep['day_of_week'] = df_prep['timestamp'].dt.dayofweek
df_prep['is_weekend'] = df_prep['day_of_week'].isin([5, 6]).astype(int)
df_prep['is_night_transaction'] = df_prep['hour_of_day'].isin([23, 0, 1, 2, 3, 4, 5]).astype(int)
df_prep['log_amount'] = np.log1p(df_prep['transaction_amount_usd'])
df_prep['is_new_account'] = (df_prep['account_age_days'] < 30).astype(int)
df_prep['scam_vector_count'] = (df_prep['received_fake_sms'] +
                               df_prep['received_phishing_email'] +
                               df_prep['received_fraud_call'])

# Define target and features
X = df_prep.drop(columns=['transaction_id', 'timestamp', 'customer_id', 'device_ip', 'is_fraud'])
y = df_prep['is_fraud']

# Identify column types
numeric_features = ['account_age_days', 'transaction_amount_usd', 'log_amount', 'hour_of_day', 'day_of_week',
                    'unusual_delay_flag', 'received_fake_sms', 'received_phishing_email', 'received_fraud_call',
                    'is_weekend', 'is_night_transaction', 'scam_vector_count', 'is_new_account']
categorical_features = ['payment_method', 'channel', 'merchant_category', 'location_country']

# 2. Build Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# Fit and transform dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_proc = preprocessor.fit_transform(X_train)

# Retrieve exact feature names from the pipeline
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + encoded_cat_features

# 3. Train the XGBoost model
model = XGBClassifier(n_estimators=100, max_depth=5, random_state=42, eval_metric='logloss')
model.fit(X_train_proc, y_train)

print("Model training complete! Variable definitions for 'model', 'preprocessor', and 'all_feature_names' are active.")

Model training complete! Variable definitions for 'model', 'preprocessor', and 'all_feature_names' are active.


In [7]:
import numpy as np
import pandas as pd
import shap
from xgboost import XGBClassifier

# -----------------------------------------------------------------------------
# 1. SHAP EXPLAINER SETUP (Run once after training your model)
# -----------------------------------------------------------------------------
# Initialize SHAP TreeExplainer on your trained XGBoost model
explainer = shap.TreeExplainer(model)


# -----------------------------------------------------------------------------
# 2. FUNCTION TO CALCULATE INDIVIDUAL RISK & FEATURE BREAKDOWN
# -----------------------------------------------------------------------------
def analyze_individual_transaction_risk(raw_txn_df, processed_features, feature_names):
    """
    Computes overall fraud risk percentage and exact individual factor contributions.
    """
    # Get total predicted risk percentage (0% to 100%)
    overall_risk_pct = model.predict_proba(processed_features)[0][1] * 100

    # Calculate SHAP values (log-odds contribution per feature)
    shap_vals = explainer.shap_values(processed_features)[0]

    # Focus on key features user wants to inspect
    df_factors = pd.DataFrame({
        'Factor': feature_names,
        'Shap_Value': shap_vals
    })

    # Group one-hot encoded categories back or highlight primary factors
    df_factors['Absolute_Impact'] = df_factors['Shap_Value'].abs()
    total_impact = df_factors['Absolute_Impact'].sum()

    # Calculate relative percentage contribution of each feature to the risk score
    if total_impact > 0:
        df_factors['Risk_Contribution_%'] = (df_factors['Absolute_Impact'] / total_impact) * 100
    else:
        df_factors['Risk_Contribution_%'] = 0.0

    # Sort factors by impact
    df_factors = df_factors.sort_values(by='Risk_Contribution_%', ascending=False)

    return round(overall_risk_pct, 2), df_factors


# -----------------------------------------------------------------------------
# 3. EXAMPLE INFERENCE FOR A SINGLE CUSTOMER / TRANSACTION
# -----------------------------------------------------------------------------
# Raw sample transaction data
sample_txn = pd.DataFrame([{
    'account_age_days': 10,                 # Same/New account indicator
    'transaction_amount_usd': 3200.00,
    'log_amount': np.log1p(3200.00),
    'unusual_delay_flag': 1,               # Late/Delayed transaction
    'received_fake_sms': 1,                # Fake SMS factor
    'received_phishing_email': 1,          # Misleading Email factor
    'received_fraud_call': 0,
    'hour_of_day': 2,                      # Transaction Time
    'day_of_week': 5,                      # Transaction Date/Day
    'is_weekend': 1,
    'is_night_transaction': 1,             # Transaction Time factor
    'scam_vector_count': 2,
    'is_new_account': 1,                   # Account tenure factor
    'payment_method': 'WIRE_TRANSFER',
    'channel': 'WEB_PORTAL',
    'merchant_category': 'CRYPTO_EXCHANGE',
    'location_country': 'NIGERIA'
}])

# Process features through preprocessor pipeline
sample_processed = preprocessor.transform(sample_txn)

# Run risk analysis
risk_percentage, factor_breakdown = analyze_individual_transaction_risk(
    sample_txn, sample_processed, all_feature_names
)

# -----------------------------------------------------------------------------
# 4. OUTPUT RESULTS
# -----------------------------------------------------------------------------
print(f"==================================================")
print(f" TOTAL PREDICTED FRAUD RISK SCORE: {risk_percentage}%")
print(f"==================================================")
print("\nINDIVIDUAL FACTOR RISK BREAKDOWN:")
print("--------------------------------------------------")

# Filter and format key specific factors requested
for idx, row in factor_breakdown.iterrows():
    factor_name = row['Factor']
    contrib = row['Risk_Contribution_%']

    # Map raw features to human-readable names
    if 'received_fake_sms' in factor_name:
        print(f"• Fake SMS/Smishing Flag:         {contrib:.2f}% of risk decision")
    elif 'received_phishing_email' in factor_name:
        print(f"• Misleading/Phishing Email:      {contrib:.2f}% of risk decision")
    elif 'unusual_delay_flag' in factor_name:
        print(f"• Late/Delayed Transaction:       {contrib:.2f}% of risk decision")
    elif 'hour_of_day' in factor_name or 'is_night_transaction' in factor_name:
        print(f"• Transaction Time (Hour/Night):  {contrib:.2f}% of risk decision")
    elif 'account_age_days' in factor_name or 'is_new_account' in factor_name:
        print(f"• Account History/Same Account:   {contrib:.2f}% of risk decision")
    elif 'transaction_amount_usd' in factor_name:
        print(f"• Transaction Amount (USD):       {contrib:.2f}% of risk decision")

 TOTAL PREDICTED FRAUD RISK SCORE: 90.58999633789062%

INDIVIDUAL FACTOR RISK BREAKDOWN:
--------------------------------------------------
• Transaction Amount (USD):       45.95% of risk decision
• Account History/Same Account:   13.54% of risk decision
• Fake SMS/Smishing Flag:         5.31% of risk decision
• Misleading/Phishing Email:      4.66% of risk decision
• Transaction Time (Hour/Night):  3.89% of risk decision
• Late/Delayed Transaction:       2.17% of risk decision
• Transaction Time (Hour/Night):  0.91% of risk decision
• Account History/Same Account:   0.16% of risk decision


In [9]:
# 1. Apply the preprocessing pipeline to the entire dataset features
X_all_proc = preprocessor.transform(X)

# 2. Predict fraud probabilities (column index 1 represents the probability of class '1' - Fraud)
df_prep['fraud_risk_score_%'] = model.predict_proba(X_all_proc)[:, 1] * 100

# 3. Sort the dataset to find the top 10 transactions with the highest risk scores
top_10_high_risk = df_prep.sort_values(by='fraud_risk_score_%', ascending=False).head(10)

# Display selected columns for readability
columns_to_show = [
    'transaction_id', 'customer_id', 'transaction_amount_usd',
    'account_age_days', 'payment_method', 'location_country',
    'scam_vector_count', 'is_fraud', 'fraud_risk_score_%'
]

display(top_10_high_risk[columns_to_show])

,transaction_id,customer_id,transaction_amount_usd,account_age_days,payment_method,location_country,scam_vector_count,is_fraud,fraud_risk_score_%
3735,TXN_103736,CUST_36314,914.03,178,WIRE_TRANSFER,UK,2,1,99.968407
3418,TXN_103419,CUST_86935,466.79,293,WIRE_TRANSFER,INDIA,1,1,99.945419
4175,TXN_104176,CUST_53638,6652.93,473,DEBIT_CARD,RUSSIA,1,1,99.912064
5052,TXN_105053,CUST_70140,396.63,67,DEBIT_CARD,UK,2,1,99.910744
3997,TXN_103998,CUST_94973,752.18,96,WIRE_TRANSFER,ROMANIA,1,1,99.908356
4380,TXN_104381,CUST_48279,226.86,201,WIRE_TRANSFER,ROMANIA,2,1,99.904007
9727,TXN_109728,CUST_63662,424.52,738,CRYPTO_TRANSFER,UK,1,1,99.895081
2535,TXN_102536,CUST_65861,2305.97,91,CREDIT_CARD,ROMANIA,2,1,99.893967
3986,TXN_103987,CUST_71740,270.26,1504,WIRE_TRANSFER,UK,2,1,99.882439
9108,TXN_109109,CUST_76798,452.24,396,WIRE_TRANSFER,RUSSIA,2,1,99.869255


give me the top 10 high risk financial fraud in this data set
I will calculate the predicted fraud probabilities for all transactions in our dataset using our trained XGBoost model and display the top 10 transactions with the highest risk scores.

Here are the top 10 highest-risk financial transactions detected in the dataset by our model, ranked by their predicted risk score:

Transaction ID: TXN_103736 (Customer: CUST_36314)

Risk Score: 99.97%
Amount: $914.03 | Payment Method: WIRE_TRANSFER | Country: UK
Flags: 2 scam indicators (e.g. fake SMS/phishing)
Transaction ID: TXN_103419 (Customer: CUST_86935)

Risk Score: 99.95%
Amount: $466.79 | Payment Method: WIRE_TRANSFER | Country: INDIA
Flags: 1 scam indicator
Transaction ID: TXN_104176 (Customer: CUST_53638)

Risk Score: 99.91%
Amount: $6,652.93 | Payment Method: DEBIT_CARD | Country: RUSSIA
Flags: 1 scam indicator
Transaction ID: TXN_105053 (Customer: CUST_70140)

Risk Score: 99.91%
Amount: $396.63 | Payment Method: DEBIT_CARD | Country: UK
Flags: 2 scam indicators
Transaction ID: TXN_103998 (Customer: CUST_94973)

Risk Score: 99.91%
Amount: $752.18 | Payment Method: WIRE_TRANSFER | Country: ROMANIA
Flags: 1 scam indicator
Transaction ID: TXN_104381 (Customer: CUST_48279)

Risk Score: 99.90%
Amount: $226.86 | Payment Method: WIRE_TRANSFER | Country: ROMANIA
Flags: 2 scam indicators
Transaction ID: TXN_109728 (Customer: CUST_63662)

Risk Score: 99.90%
Amount: $424.52 | Payment Method: CRYPTO_TRANSFER | Country: UK
Flags: 1 scam indicator
Transaction ID: TXN_102536 (Customer: CUST_65861)

Risk Score: 99.89%
Amount: $2,305.97 | Payment Method: CREDIT_CARD | Country: ROMANIA
Flags: 2 scam indicators
Transaction ID: TXN_103987 (Customer: CUST_71740)

Risk Score: 99.88%
Amount: $270.26 | Payment Method: WIRE_TRANSFER | Country: UK
Flags: 2 scam indicators
Transaction ID: TXN_109109 (Customer: CUST_76798)

Risk Score: 99.87%
Amount: $452.24 | Payment Method: WIRE_TRANSFER | Country: RUSSIA
Flags: 2 scam indicators